# Speedup case study: one vector multiplication, six ways

The notebook times `a * b` for two vectors of 1,024 to about a million numbers:

- **Python loop:** one element at a time, interpreted. This is the baseline.
- **Numba:** the same loop, compiled to machine code by `@njit`. With `fastmath=True` the compiler may use SIMD
  instructions (SSE, AVX) that multiply several numbers in one instruction.
- **Numba parallel:** the compiled loop split over all CPU cores (`parallel=True`, `prange`).
- **NumPy:** `np.multiply`, compiled C that also uses SIMD.
- **GPU, copy each call:** PyTorch copies both vectors to the GPU, multiplies, and leaves the result there.
- **GPU, data on device:** the vectors already live on the GPU.

It runs on any CPU (the course's `pytorch` container). The two GPU lines appear only when PyTorch finds CUDA: on
Colab, pick a GPU runtime (Runtime > Change runtime type). `COURSE_SMOKE=1` (the course's smoke test) stops at
65,536 elements.

In [ ]:
import os
import time

import matplotlib.pyplot as plt
import numpy as np
import torch
from numba import njit, prange

SMOKE = os.environ.get("COURSE_SMOKE") == "1"
sizes = [2**i for i in range(10, 17 if SMOKE else 21)]  # 1,024 to 65,536 or 1,048,576 elements
REPEATS = 10

gpu = torch.cuda.is_available()
device = torch.device("cuda" if gpu else "cpu")
print("GPU:", torch.cuda.get_device_name(0) if gpu else "none, so the GPU lines are left out")
print("CPU cores:", os.cpu_count())


def best_time(f, *args):
    """The fastest of REPEATS calls, in seconds, after one warm-up call (which also compiles Numba code)."""
    f(*args)
    times = []
    for _ in range(REPEATS):
        start = time.perf_counter()
        f(*args)
        if gpu:
            torch.cuda.synchronize()  # a GPU call returns before the GPU has finished
        times.append(time.perf_counter() - start)
    return min(times)

In [ ]:
def python_loop(a, b):
    out = np.empty_like(a)
    for i in range(len(a)):
        out[i] = a[i] * b[i]
    return out


@njit(fastmath=True)  # the same loop, compiled; fastmath lets it use SIMD instructions
def numba_loop(a, b):
    out = np.empty_like(a)
    for i in range(len(a)):
        out[i] = a[i] * b[i]
    return out


@njit(fastmath=True, parallel=True)  # and split over the CPU cores
def numba_parallel(a, b):
    out = np.empty_like(a)
    for i in prange(len(a)):
        out[i] = a[i] * b[i]
    return out


def numpy_multiply(a, b):
    return np.multiply(a, b)


def gpu_copy_each_call(a, b):  # copy both vectors to the GPU on every call
    return torch.from_numpy(a).to(device) * torch.from_numpy(b).to(device)


def gpu_data_on_device(a_gpu, b_gpu):  # the vectors are already on the GPU
    return a_gpu * b_gpu

In [ ]:
methods = {"Python loop": python_loop, "Numba": numba_loop, "Numba parallel": numba_parallel,
           "NumPy": numpy_multiply}
if gpu:
    methods["GPU, copy each call"] = gpu_copy_each_call
times = {name: [] for name in methods} | ({"GPU, data on device": []} if gpu else {})

rng = np.random.default_rng(0)
for n in sizes:
    a, b = rng.random(n), rng.random(n)
    for name, f in methods.items():
        times[name].append(best_time(f, a, b))
    if gpu:
        times["GPU, data on device"].append(
            best_time(gpu_data_on_device, torch.from_numpy(a).to(device), torch.from_numpy(b).to(device)))

In [ ]:
plt.figure(figsize=(10, 6))
for name, t in times.items():
    plt.plot(sizes, t, marker="o", label=name)
plt.xscale("log", base=2)
plt.yscale("log")
plt.xlabel("Vector size (elements)")
plt.ylabel(f"Time per call (seconds, best of {REPEATS})")
plt.title("Multiplying two vectors: compiled, parallel and GPU code against a Python loop")
plt.legend()
plt.grid(True, which="both", alpha=0.3)
plt.show()

for i in (0, -1):
    print(f"Speedup over the Python loop at {sizes[i]:,} elements:")
    for name, t in times.items():
        print(f"  {name:20s} {times['Python loop'][i] / t[i]:9,.1f}x")

## What to look for

The numbers depend on the machine, so compare the shapes of the lines.

- Compiling the loop (Numba) or calling compiled code (NumPy) makes it hundreds of times faster than the Python
  loop. The arithmetic is the same: the Python loop pays for the interpreter on every element.
- Numba and NumPy end up close, because both run a tight compiled loop with SIMD instructions.
- Numba parallel wakes its threads on every call, which can cost more than the whole multiplication. A multiply
  also does so little work per number that memory bandwidth, not the cores, limits it. Parallel loops pay off when
  each element needs real work.
- On a GPU, copying the vectors over takes longer than the multiplication. With the data already on the device,
  the GPU wins on large vectors. Keep the data where it is computed.